# tRNA Denovo Design

Evaluate the quality of 200 tRNA sequences generated by the model in a zero-shot setting.  
Two metrics are used:
- **Sequence Similarity**: global pairwise alignment against 32 natural tRNA reference structures in RSCB
- **TM-score**: structural similarity of predicted 3D structures vs natural tRNA

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager
from Bio import pairwise2

try:
    import design_paths as DP
except ModuleNotFoundError:
    from notebooks.design import design_paths as DP

TRNA_DATA_DIR = DP.data_dir() / "tRNA"

# Font setup
FONT_PATH = str(DP.font_path())
try:
    font_manager.fontManager.addfont(FONT_PATH)
    plt.rcParams["font.sans-serif"] = ["Arial"]
except:
    plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["axes.linewidth"] = 2.0

# Data paths
TRNA_FASTA = TRNA_DATA_DIR / "tRNA_200.fasta"
TRNA_REF_DIR = TRNA_DATA_DIR / "tRNA_ref"
SEQ_SIM_FILE = TRNA_DATA_DIR / "sequence_similarity.txt"
TM_SCORE_FILE = TRNA_DATA_DIR / "tm-score.txt"

# Figure saving
PIC_DIR = str(DP.figures_dir())
os.makedirs(PIC_DIR, exist_ok=True)

def save_fig(name):
    """Save current figure as 300dpi PNG and SVG with transparent background."""
    plt.savefig(os.path.join(PIC_DIR, f"{name}.png"), format='png', dpi=300,
                bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(PIC_DIR, f"{name}.svg"), format='svg', dpi=300,
                bbox_inches='tight', transparent=True)

## 1. Sequence Similarity Computation

For each generated tRNA, compute global pairwise alignment (Needleman-Wunsch, no gap penalty) against all 32 reference tRNA sequences extracted from PDB CIF files. Report the maximum similarity.

In [ ]:
def parse_cif_sequence(cif_path: str) -> str:
    """Extract RNA sequence from CIF file."""
    with open(cif_path, 'r') as f:
        content = f.read()
    pattern = r'_entity_poly\.pdbx_seq_one_letter_code_can\s+(\S+)'
    match = re.search(pattern, content)
    if match:
        seq = match.group(1).strip().strip('"\'')
        return ''.join(c for c in seq.upper() if c in 'ACGU')
    return ""


def similarity_globalxx(seq1: str, seq2: str) -> float:
    """Compute sequence similarity via global alignment."""
    aln = pairwise2.align.globalxx(seq1, seq2, one_alignment_only=True)[0]
    a, b = aln.seqA, aln.seqB
    matches = sum(x == y for x, y in zip(a, b))
    return matches / len(a) if len(a) else 0.0


def load_fasta(fasta_path: str) -> list:
    """Parse FASTA file, return [(name, seq), ...]."""
    sequences = []
    name, seq = None, []
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if name is not None:
                    sequences.append((name, ''.join(seq)))
                name, seq = line[1:], []
            else:
                seq.append(line.upper())
    if name is not None:
        sequences.append((name, ''.join(seq)))
    return sequences


# Load reference sequences from CIF files
ref_seqs = {}
for fn in os.listdir(TRNA_REF_DIR):
    if fn.endswith('.cif'):
        seq = parse_cif_sequence(os.path.join(TRNA_REF_DIR, fn))
        if seq:
            ref_seqs[fn] = seq
print(f"Loaded {len(ref_seqs)} reference tRNA sequences")

# Load generated sequences
gen_seqs = load_fasta(TRNA_FASTA)
print(f"Loaded {len(gen_seqs)} generated sequences")

# Compute max similarity for each generated sequence
similarities = []
for i, (name, seq) in enumerate(gen_seqs):
    max_sim = max(similarity_globalxx(seq, rs) for rs in ref_seqs.values())
    similarities.append(max_sim)
    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(gen_seqs)}")

print(f"\nSequence similarity range: {min(similarities):.4f} - {max(similarities):.4f}")
print(f"Mean: {np.mean(similarities):.4f}, Median: {np.median(similarities):.4f}")

## 2. Violin Plot — Sequence Similarity

In [ ]:
# Load pre-computed similarity (or use computed values above)
with open(SEQ_SIM_FILE) as f:
    sim_values = [float(line.strip()) for line in f if line.strip()]

df_sim = pd.DataFrame({'Sequence Similarity': sim_values, 'Category': 'Generated tRNA'})
print(f"Count: {len(sim_values)}, Range: {min(sim_values):.3f}-{max(sim_values):.3f}, "
      f"Mean: {np.mean(sim_values):.3f}, Median: {np.median(sim_values):.3f}")

plt.figure(figsize=(5, 6))
sns.set_style("ticks")
ax = sns.violinplot(data=df_sim, x="Category", y="Sequence Similarity",
                    color="#1f77b4", inner="box", linewidth=1.5, cut=0)
plt.title("Sequence Similarity of Generated tRNA\nvs Natural tRNA", fontsize=18, pad=20)
plt.ylabel("Sequence Similarity", fontsize=16)
plt.xlabel("")
ax.tick_params(axis='both', which='major', labelsize=14, width=2.0, length=6)
plt.ylim(0, 1.0)
sns.despine()
plt.tight_layout()
save_fig("tRNA_seq_similarity")
plt.show()

## 3. Violin Plot — TM-score

In [ ]:
with open(TM_SCORE_FILE) as f:
    tm_scores = [float(line.strip()) for line in f if line.strip()]

df_tm = pd.DataFrame({'TM-score': tm_scores, 'Category': 'Generated tRNA'})
print(f"Count: {len(tm_scores)}, Range: {min(tm_scores):.3f}-{max(tm_scores):.3f}, "
      f"Mean: {np.mean(tm_scores):.3f}, Median: {np.median(tm_scores):.3f}")

plt.figure(figsize=(5, 6))
sns.set_style("ticks")
ax = sns.violinplot(data=df_tm, x="Category", y="TM-score",
                    color="#FFC994", inner="box", linewidth=1.5, cut=0)
plt.title("TM-score of Generated tRNA vs Natural tRNA", fontsize=18, pad=20)
plt.ylabel("TM-score", fontsize=16)
plt.xlabel("")
ax.tick_params(axis='both', which='major', labelsize=14, width=2.0, length=6)
plt.ylim(0, 1.0)
sns.despine()
plt.tight_layout()
save_fig("tRNA_tm_score")
plt.show()